# Analyzing Subliminal Learning Results with Inspect

This notebook demonstrates how to analyze evaluation results from Inspect, including loading logs, extracting metrics, and creating visualizations.

In [ ]:
# Setup
import sys
sys.path.append('..')

import json
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict

from inspect_ai.log import read_eval_log, EvalLog
from loguru import logger

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Loading Evaluation Logs

Inspect stores detailed logs of all evaluations. Let's load and explore them:

In [ ]:
def load_latest_log(log_dir: str, pattern: str = "*.json") -> EvalLog:
    """Load the most recent evaluation log from a directory."""
    log_path = Path(log_dir)
    if not log_path.exists():
        raise FileNotFoundError(f"Log directory not found: {log_dir}")
    
    # Find all log files
    log_files = list(log_path.glob(pattern))
    if not log_files:
        raise FileNotFoundError(f"No log files found in {log_dir}")
    
    # Get most recent
    latest_log = max(log_files, key=lambda p: p.stat().st_mtime)
    logger.info(f"Loading log: {latest_log}")
    
    return read_eval_log(str(latest_log))

# Example: Load an evaluation log
try:
    log = load_latest_log("./inspect_logs/owl_preference")
    print(f"Loaded evaluation: {log.eval.task}")
    print(f"Model: {log.eval.model}")
    print(f"Samples: {len(log.samples)}")
    print(f"Metrics: {list(log.results.metrics.keys())}")
except FileNotFoundError as e:
    print(f"No logs found: {e}")
    print("Run some evaluations first!")

## 2. Extracting Key Metrics

Let's extract and analyze the key metrics from evaluation logs:

In [ ]:
def extract_metrics(log: EvalLog) -> Dict[str, Any]:
    """Extract key metrics from an evaluation log."""
    metrics = {
        "model": log.eval.model,
        "task": log.eval.task,
        "samples": len(log.samples),
        "accuracy": log.results.metrics.get("accuracy", {}).get("value", 0),
        "stderr": log.results.metrics.get("accuracy", {}).get("stderr", 0),
    }
    
    # Add any custom metrics
    for metric_name, metric_data in log.results.metrics.items():
        if metric_name not in ["accuracy", "stderr"]:
            metrics[metric_name] = metric_data.get("value", 0)
    
    return metrics

def analyze_responses(log: EvalLog) -> Dict[str, int]:
    """Analyze the distribution of responses."""
    responses = []
    
    for sample in log.samples:
        if sample.output and sample.output.completion:
            # Normalize response
            response = sample.output.completion.strip().lower()
            responses.append(response)
    
    return Counter(responses)

# Example analysis
try:
    log = load_latest_log("./inspect_logs/owl_preference")
    
    # Extract metrics
    metrics = extract_metrics(log)
    print("\nEvaluation Metrics:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")
    
    # Analyze responses
    response_dist = analyze_responses(log)
    print("\nTop 5 Responses:")
    for response, count in response_dist.most_common(5):
        print(f"  '{response}': {count} ({count/len(log.samples):.1%})")
        
except FileNotFoundError:
    print("No logs to analyze")

## 3. Comparing Multiple Models

Compare results across different models and training methods:

In [ ]:
def compare_models(log_dirs: Dict[str, str]) -> pd.DataFrame:
    """Compare evaluation results across multiple models."""
    results = []
    
    for model_name, log_dir in log_dirs.items():
        try:
            log = load_latest_log(log_dir)
            metrics = extract_metrics(log)
            metrics["model_name"] = model_name
            results.append(metrics)
        except Exception as e:
            logger.warning(f"Failed to load {model_name}: {e}")
    
    return pd.DataFrame(results)

# Example comparison
model_logs = {
    "Baseline": "./inspect_logs/baseline",
    "SFT Student": "./inspect_logs/sft_student",
    "RL Student": "./inspect_logs/rl_student",
    "DPO Student": "./inspect_logs/dpo_student"
}

try:
    comparison_df = compare_models(model_logs)
    if not comparison_df.empty:
        print("Model Comparison:")
        print(comparison_df[["model_name", "accuracy", "samples"]].to_string(index=False))
except Exception as e:
    print(f"Could not compare models: {e}")

## 4. Visualizing Results

Create visualizations to better understand the results:

In [ ]:
def plot_trait_transmission(results_df: pd.DataFrame, target_trait: str = "owl"):
    """Plot trait transmission rates across models."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Sort by accuracy
    results_df = results_df.sort_values('accuracy')
    
    # Create bar plot
    bars = ax.bar(results_df['model_name'], results_df['accuracy'] * 100)
    
    # Color bars based on performance
    for i, (idx, row) in enumerate(results_df.iterrows()):
        if row['accuracy'] > 0.05:  # Above 5% threshold
            bars[i].set_color('green')
        else:
            bars[i].set_color('gray')
    
    # Add threshold line
    ax.axhline(y=5, color='red', linestyle='--', label='5% Threshold')
    
    # Formatting
    ax.set_ylabel('Preference Rate (%)')
    ax.set_title(f'{target_trait.capitalize()} Preference Transmission Across Models')
    ax.set_ylim(0, max(results_df['accuracy'] * 100) * 1.2)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}%', ha='center', va='bottom')
    
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Example visualization
sample_data = pd.DataFrame([
    {"model_name": "Baseline", "accuracy": 0.02},
    {"model_name": "SFT Student", "accuracy": 0.15},
    {"model_name": "RL Student", "accuracy": 0.08},
    {"model_name": "DPO Student", "accuracy": 0.12}
])

plot_trait_transmission(sample_data)

In [ ]:
def plot_response_distribution(log: EvalLog, top_n: int = 10):
    """Plot the distribution of model responses."""
    response_dist = analyze_responses(log)
    
    # Get top N responses
    top_responses = response_dist.most_common(top_n)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    responses = [r[0] for r in top_responses]
    counts = [r[1] for r in top_responses]
    
    bars = ax.bar(responses, counts)
    
    # Highlight target response
    for i, response in enumerate(responses):
        if response == "owl":
            bars[i].set_color('green')
    
    ax.set_xlabel('Response')
    ax.set_ylabel('Count')
    ax.set_title(f'Top {top_n} Model Responses')
    
    # Add percentage labels
    total = len(log.samples)
    for bar, count in zip(bars, counts):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{count/total:.1%}', ha='center', va='bottom')
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Example (would need actual log)
# plot_response_distribution(log)

## 5. Statistical Analysis

Perform statistical tests to validate results:

In [ ]:
from scipy import stats

def statistical_comparison(log1: EvalLog, log2: EvalLog, target_trait: str = "owl"):
    """Perform statistical comparison between two models."""
    
    # Extract binary outcomes (1 if matches target, 0 otherwise)
    def get_outcomes(log):
        outcomes = []
        for sample in log.samples:
            if sample.output and sample.output.completion:
                response = sample.output.completion.strip().lower()
                outcomes.append(1 if target_trait in response else 0)
        return outcomes
    
    outcomes1 = get_outcomes(log1)
    outcomes2 = get_outcomes(log2)
    
    # Calculate rates
    rate1 = sum(outcomes1) / len(outcomes1)
    rate2 = sum(outcomes2) / len(outcomes2)
    
    # Perform chi-square test
    contingency_table = [
        [sum(outcomes1), len(outcomes1) - sum(outcomes1)],
        [sum(outcomes2), len(outcomes2) - sum(outcomes2)]
    ]
    
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
    
    # Calculate confidence intervals
    ci1 = stats.binom.interval(0.95, len(outcomes1), rate1)
    ci2 = stats.binom.interval(0.95, len(outcomes2), rate2)
    
    print("Statistical Comparison:")
    print(f"\nModel 1: {log1.eval.model}")
    print(f"  Rate: {rate1:.3f} ({ci1[0]/len(outcomes1):.3f}, {ci1[1]/len(outcomes1):.3f})")
    print(f"\nModel 2: {log2.eval.model}")
    print(f"  Rate: {rate2:.3f} ({ci2[0]/len(outcomes2):.3f}, {ci2[1]/len(outcomes2):.3f})")
    print(f"\nDifference: {rate2 - rate1:+.3f}")
    print(f"Chi-square test: χ² = {chi2:.3f}, p = {p_value:.4f}")
    print(f"Significant at α=0.05: {'Yes' if p_value < 0.05 else 'No'}")
    
    return {
        "rate1": rate1,
        "rate2": rate2,
        "difference": rate2 - rate1,
        "p_value": p_value,
        "significant": p_value < 0.05
    }

# Example usage (would need actual logs)
# baseline_log = load_latest_log("./inspect_logs/baseline")
# student_log = load_latest_log("./inspect_logs/sft_student")
# results = statistical_comparison(baseline_log, student_log)

## 6. Experiment Summary Report

Generate a comprehensive report of your experiment:

In [ ]:
def generate_experiment_report(experiment_name: str, log_dirs: Dict[str, str]):
    """Generate a comprehensive experiment report."""
    
    report = {
        "experiment": experiment_name,
        "timestamp": pd.Timestamp.now().isoformat(),
        "models": {},
        "summary": {}
    }
    
    # Collect data from all models
    baseline_rate = None
    best_model = None
    best_rate = 0
    
    for model_name, log_dir in log_dirs.items():
        try:
            log = load_latest_log(log_dir)
            metrics = extract_metrics(log)
            response_dist = analyze_responses(log)
            
            model_data = {
                "accuracy": metrics["accuracy"],
                "samples": metrics["samples"],
                "top_response": response_dist.most_common(1)[0] if response_dist else ("none", 0),
                "unique_responses": len(response_dist)
            }
            
            report["models"][model_name] = model_data
            
            # Track baseline and best
            if "baseline" in model_name.lower():
                baseline_rate = metrics["accuracy"]
            elif metrics["accuracy"] > best_rate:
                best_model = model_name
                best_rate = metrics["accuracy"]
                
        except Exception as e:
            logger.warning(f"Failed to process {model_name}: {e}")
    
    # Calculate summary statistics
    if baseline_rate is not None and best_model:
        report["summary"] = {
            "best_model": best_model,
            "best_accuracy": best_rate,
            "baseline_accuracy": baseline_rate,
            "improvement": best_rate - baseline_rate,
            "trait_transmitted": best_rate > baseline_rate + 0.05
        }
    
    # Print report
    print("\n" + "="*60)
    print(f"EXPERIMENT REPORT: {experiment_name}")
    print("="*60)
    
    print("\nModel Performance:")
    for model_name, data in report["models"].items():
        print(f"\n{model_name}:")
        print(f"  Accuracy: {data['accuracy']:.3f}")
        print(f"  Top Response: '{data['top_response'][0]}' ({data['top_response'][1]} times)")
        print(f"  Unique Responses: {data['unique_responses']}")
    
    if report["summary"]:
        print("\nSummary:")
        print(f"  Best Model: {report['summary']['best_model']}")
        print(f"  Improvement over baseline: {report['summary']['improvement']:+.3f}")
        print(f"  Trait Successfully Transmitted: {report['summary']['trait_transmitted']}")
    
    # Save report
    report_path = f"./results/{experiment_name}_report.json"
    Path("./results").mkdir(exist_ok=True)
    
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"\nReport saved to: {report_path}")
    
    return report

# Example report generation
# report = generate_experiment_report(
#     "owl_preference_transmission",
#     {
#         "Baseline": "./inspect_logs/baseline",
#         "SFT Student": "./inspect_logs/sft_student",
#         "RL Student": "./inspect_logs/rl_student",
#         "DPO Student": "./inspect_logs/dpo_student"
#     }
# )

## 7. Advanced Analysis: Sample-Level Investigation

Dive deep into individual samples to understand model behavior:

In [ ]:
def analyze_sample_patterns(log: EvalLog, target_trait: str = "owl"):
    """Analyze patterns in individual samples."""
    
    # Group samples by outcome
    correct_samples = []
    incorrect_samples = []
    
    for sample in log.samples:
        if sample.output and sample.output.completion:
            response = sample.output.completion.strip().lower()
            if target_trait in response:
                correct_samples.append(sample)
            else:
                incorrect_samples.append(sample)
    
    print(f"Sample Analysis for '{target_trait}' preference:")
    print(f"\nCorrect samples: {len(correct_samples)} ({len(correct_samples)/len(log.samples):.1%})")
    print(f"Incorrect samples: {len(incorrect_samples)} ({len(incorrect_samples)/len(log.samples):.1%})")
    
    # Analyze prompts that worked well
    print("\nPrompts with highest success rate:")
    prompt_success = defaultdict(lambda: {"correct": 0, "total": 0})
    
    for sample in log.samples:
        if sample.input and len(sample.input) > 0:
            prompt = sample.input[-1].content  # Get user message
            prompt_success[prompt]["total"] += 1
            
            if sample.output and sample.output.completion:
                response = sample.output.completion.strip().lower()
                if target_trait in response:
                    prompt_success[prompt]["correct"] += 1
    
    # Sort by success rate
    prompt_rates = [
        (prompt, data["correct"] / data["total"], data["total"])
        for prompt, data in prompt_success.items()
    ]
    prompt_rates.sort(key=lambda x: x[1], reverse=True)
    
    for prompt, rate, total in prompt_rates[:5]:
        print(f"\n  Prompt: \"{prompt[:50]}...\"")
        print(f"  Success rate: {rate:.1%} ({total} samples)")
    
    # Show example correct/incorrect responses
    print("\n\nExample CORRECT responses:")
    for sample in correct_samples[:3]:
        print(f"  - \"{sample.output.completion.strip()}\"")
    
    print("\nExample INCORRECT responses:")
    for sample in incorrect_samples[:3]:
        print(f"  - \"{sample.output.completion.strip()}\"")

# Example usage
# analyze_sample_patterns(log)

## Summary

This notebook demonstrated how to:

1. **Load and explore** Inspect evaluation logs
2. **Extract metrics** and analyze response distributions
3. **Compare models** across different training methods
4. **Visualize results** with meaningful plots
5. **Perform statistical tests** to validate findings
6. **Generate comprehensive reports** for experiments
7. **Investigate individual samples** to understand model behavior

## Best Practices for Analysis

1. **Always compare to baseline**: Trait transmission is relative to base model behavior
2. **Use sufficient samples**: At least 200 for reliable statistics
3. **Check controls**: Verify shuffled/cross-model controls show no transmission
4. **Consider variance**: Run multiple evaluations to account for randomness
5. **Document everything**: Use Inspect's metadata features to track experiment details

## Next Steps

- Use `inspect view` for interactive exploration of results
- Export data for further analysis in specialized tools
- Share Inspect logs for reproducibility
- Extend scorers and analysis for new experiment types